# py-tradeSeq — Tutorial on synthetic trajectory data

Walkthrough of every public function in `pytradeseq` on a small simulated dataset.

## 1. What this package does

`pytradeseq` is a Python port of [tradeSeq](https://github.com/statOmics/tradeSeq) (Van den Berge et al. *Nature Comm* 2020) — **trajectory-based differential expression** for single-cell RNA-seq. Given a fitted Slingshot trajectory + counts matrix, fits a **negative-binomial generalized additive model (NB-GAM)** per gene per lineage, then performs Wald tests for various inference questions:

- `associationTest` — is expression associated with pseudotime?
- `startVsEndTest` — does expression differ between trajectory endpoints?
- `diffEndTest` — do lineage endpoints differ from each other?
- `patternTest` — does the expression SHAPE differ across lineages?
- `earlyDETest` — restricted patternTest on early pseudotime

GAM backbone: **`statsmodels.gam.GLMGam`** with NB family + cubic B-spline smoothers. Known to differ from R's `mgcv` at small df; see [`MATH.md`](../MATH.md).

## 2. Install + import

In [1]:
import sys
sys.path.insert(0, '/scratch/users/steorra/analysis/omicverse_traj_dev/py-tradeSeq')
import pytradeseq
print(f'pytradeseq version: {pytradeseq.__version__}')

pytradeseq version: 0.1.0


## 3. Synthetic data

In [2]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
n_cells, n_genes, n_lin = 80, 20, 2
# Two lineages diverging from a common start
pt = np.zeros((n_cells, n_lin)); cw = np.zeros((n_cells, n_lin))
for i in range(n_cells):
    L = rng.integers(0, n_lin)
    pt[i, L] = rng.uniform(0, 1)
    cw[i, L] = 1.0
# Counts: first 5 genes lineage-1 specific, next 5 lineage-2 specific
counts = rng.negative_binomial(5, 0.5, (n_genes, n_cells)).astype(float)
for g in range(5):
    counts[g] = rng.negative_binomial(5 + 8*pt[:,0], 0.5)
for g in range(5, 10):
    counts[g] = rng.negative_binomial(5 + 8*pt[:,1], 0.5)
print(f"counts: {counts.shape}, pseudotime: {pt.shape}, cellWeights: {cw.shape}")

counts: (20, 80), pseudotime: (80, 2), cellWeights: (80, 2)


## 4. Public functions

### 4.1 `fitGAM` — fit NB-GAM per gene per lineage

In [3]:
gams = pytradeseq.fitGAM(counts, pseudotime=pt, cellWeights=cw, nknots=4,
                          verbose=False, seed=42)
print(f"converged: {gams.converged.sum()}/{gams.converged.size}")
print(f"pcadim: not applicable; nknots={pytradeseq.nknots(gams)}")

converged: 40/40
pcadim: not applicable; nknots=4


### 4.2 `associationTest`

In [4]:
at = pytradeseq.associationTest(gams)
print(at.head(10).to_string())

          waldStat  df    pvalue  meanLogFC
Gene_1    9.674255   6  0.139057   0.969406
Gene_2    8.557537   6  0.200033   0.723457
Gene_3   15.685006   6  0.015548   0.420348
Gene_4   10.939965   6  0.090250   0.298753
Gene_5   11.106856   6  0.085130   0.697943
Gene_6    4.822736   6  0.566741   0.350222
Gene_7   13.689825   6  0.033300   0.284011
Gene_8    9.859838   6  0.130680   0.571429
Gene_9    5.329922   6  0.502246   0.304462
Gene_10   6.368795   6  0.383170   0.488982


### 4.3 `startVsEndTest`

In [5]:
svet = pytradeseq.startVsEndTest(gams)
print(svet.head(5).to_string())

        waldStat  df    pvalue  meanLogFC
Gene_1  3.081157   2  0.214257   0.632407
Gene_2  6.938554   2  0.031140   1.084645
Gene_3  2.915813   2  0.232723   0.475239
Gene_4  2.478082   2  0.289662   0.534664
Gene_5  0.859249   2  0.650753   0.302760


### 4.4 `diffEndTest` (multi-lineage)

In [6]:
det = pytradeseq.diffEndTest(gams)
print(det.head(5).to_string())

        waldStat  df    pvalue
Gene_1  0.299438   1  0.584235
Gene_2  0.248298   1  0.618276
Gene_3  0.263598   1  0.607659
Gene_4  0.459930   1  0.497657
Gene_5  0.116053   1  0.733355


### 4.5 `patternTest` — between-lineage pattern comparison

In [7]:
pt_test = pytradeseq.patternTest(gams)
print(pt_test.head(5).to_string())

           waldStat   df         pvalue
Gene_1   879.336014  100  6.836008e-125
Gene_2   388.800543  100   1.148737e-35
Gene_3   534.674449  100   1.337262e-60
Gene_4   810.181988  100  1.297219e-111
Gene_5  1328.959942  100  9.348354e-214


### 4.6 `evaluateK` — pick optimal nknots via BIC scan

In [8]:
eval_df = pytradeseq.evaluateK(counts[:10], pseudotime=pt, cellWeights=cw,
                                k_range=range(3, 7), nGenes=10, seed=42, verbose=False)
print(eval_df)

     mean_aic  mean_deviance  mean_loglik  n_fits
k                                                
3         NaN            NaN          NaN       0
4  216.226888      34.169946  -104.121255      20
5  216.941115      32.883839  -103.478202      20
6  217.552932      31.510368  -102.791466      20


## 5. Class-API: deferred to v0.2

v0.1 is functional-only. AnnData class wrapper is on the v0.2 roadmap.

## 6. Common pitfalls / FAQ

- **`statsmodels.gam` ≠ `mgcv`**: GAM fits differ at small df. Inference rankings agree (Spearman > 0.7 on associationTest) but individual p-values are NOT bit-equivalent.
- **`startVsEndTest` is approximate**: per-lineage independent fitting doesn't match R's joint-fit-with-tied-smoothness. v0.2 will fix.
- **Matrix orientation**: `counts` is `genes × cells` (R convention), `pseudotime` and `cellWeights` are `cells × lineages`.
- **`nknots` choice**: use `evaluateK` to scan; typical `nknots ∈ {4, 5, 6, 7}`.

## 7. Where to go next

- [`README.md`](../README.md), [`RECONSTRUCTION_REPORT.md`](../RECONSTRUCTION_REPORT.md), [`MATH.md`](../MATH.md)
- [`compare_R_vs_Python.ipynb`](compare_R_vs_Python.ipynb) — pipeline parity
- [`function_by_function_R_parity.ipynb`](function_by_function_R_parity.ipynb) — R⇄Py dictionary
- [tradeSeq R upstream](https://github.com/statOmics/tradeSeq)